In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("====== STEP 1: PREPARING BENCHMARK AND OFFICIAL INP-FORMER ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
INPFORMER_ROOT = Path("/kaggle/working/INP-Former")
INPFORMER_COMMIT = "17d265381d9b323a2ef6e05aab0665a85edebe84"

if not BENCHMARK_ROOT.exists():
    subprocess.run(["git", "clone", f"https://github.com/{BENCHMARK_REPOSITORY}.git", str(BENCHMARK_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(BENCHMARK_ROOT), "pull", "--ff-only"], check=True)
required_harness_file = BENCHMARK_ROOT / "few_shot" / "harness" / "runner.py"
if not required_harness_file.is_file():
    raise RuntimeError(
        "The cloned benchmark does not contain the few-shot implementation. "
        "Commit and push the local few_shot changes to BENCHMARK_REPOSITORY "
        f"before running Kaggle. Missing: {required_harness_file}"
    )

if not INPFORMER_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/luow23/INP-Former.git", str(INPFORMER_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(INPFORMER_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(INPFORMER_ROOT), "checkout", "--detach", INPFORMER_COMMIT], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(BENCHMARK_ROOT / "few_shot" / "requirements.txt")], check=True)
for import_path in (BENCHMARK_ROOT, INPFORMER_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["INPFORMER_ROOT"] = str(INPFORMER_ROOT)

# Fail before multi-GB downloads if a Python/native corruption dependency
# is unavailable or any selected operation changes image dimensions.
import numpy as np
from PIL import Image
from shared.corruption import apply_corruption
smoke_pixels = np.random.default_rng(0).integers(0, 256, (64, 96, 3), dtype=np.uint8)
smoke_image = Image.fromarray(smoke_pixels)
smoke_operations = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "motion_blur", "zoom_blur",
    "brightness", "contrast", "rotation", "zooming", "shifting",
]
for operation in smoke_operations:
    smoke_result = apply_corruption(smoke_image, operation, 1, "smoke.png", 123)
    if smoke_result.size != smoke_image.size:
        raise RuntimeError(
            f"Corruption smoke test changed dimensions for {operation}: "
            f"{smoke_image.size} -> {smoke_result.size}"
        )
print(f"Corruption smoke test passed for {len(smoke_operations)} operations.")
print("Environment ready.")


## Fixed evaluation protocol

The control block below selects the target dataset, clean evaluation, categorized or uncategorized corruptions, corruption subsets, severities, shots, device, batch size, cache, and checkpoint source. The default `DATASET_NAME = "visa"` performs the 1/2/4-shot VisA evaluations using VisA's same-dataset official checkpoints. Set `DATASET_NAME = "both"` for all six dataset/shot pairs. These controls do not alter INP-Former's official 448-resize/392-crop preprocessing, 256-pixel metric maps, Gaussian smoothing, or top-1% image score.


In [ ]:
import gc
import gdown
import torch
import zipfile

from few_shot.harness.models import (
    OFFICIAL_CHECKPOINT_URLS,
    discover_official_checkpoints,
    official_checkpoint_directory,
)
from few_shot.harness.dataset import (
    AnomalyDetectionDataset,
    build_dataset_configs,
)
from few_shot.harness.runner import run_official_evaluations

# ==============================================================================
# USER-CONTROLLABLE DATASET AND PATH SETTINGS
# ==============================================================================
# Choose "mvtec", "visa", or "both". Few-shot checkpoints are paired
# with the same target dataset; cross-dataset weights apply only to zero-shot.
DATASET_NAME = "visa"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa", "both"}:
    raise ValueError("DATASET_NAME must be 'mvtec', 'visa', or 'both'.")
DATASETS_TO_RUN = (
    ("mvtec", "visa") if DATASET_NAME == "both" else (DATASET_NAME,)
)
MVTEC_ROOT = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_ROOT = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
DATASET_ROOTS = {"mvtec": MVTEC_ROOT, "visa": VISA_ROOT}
DATASET_LABELS = {"mvtec": "MVTec AD", "visa": "VisA"}
for dataset_name in DATASETS_TO_RUN:
    dataset_root = DATASET_ROOTS[dataset_name]
    if not Path(dataset_root).is_dir():
        raise FileNotFoundError(
            f"{DATASET_LABELS[dataset_name]} is not mounted at {dataset_root}. Add the Kaggle "
            "dataset input or edit the corresponding path before downloading weights."
        )
dataset_configs = build_dataset_configs(
    mvtec_root=MVTEC_ROOT if "mvtec" in DATASETS_TO_RUN else None,
    visa_root=VISA_ROOT if "visa" in DATASETS_TO_RUN else None,
)
expected_config_names = {
    "MVTec" if name == "mvtec" else "VisA" for name in DATASETS_TO_RUN
}
if {config.name for config in dataset_configs} != expected_config_names:
    raise RuntimeError(
        f"Dataset preflight did not resolve {sorted(expected_config_names)}."
    )
for config in dataset_configs:
    sample_count = 0
    for category in config.categories:
        probe = AnomalyDetectionDataset(config=config, category=category)
        if not probe.samples:
            raise RuntimeError(
                f"Dataset preflight found no test samples for {config.name}/{category}."
            )
        missing_masks = [
            sample["sample_id"] for sample in probe.samples
            if sample.get("is_anomaly")
            and not (sample.get("mask_path") and Path(sample["mask_path"]).is_file())
        ]
        if missing_masks:
            raise RuntimeError(
                f"Dataset preflight found {len(missing_masks)} anomalous "
                f"{config.name}/{category} samples without masks; "
                f"first: {missing_masks[0]}"
            )
        sample_count += len(probe.samples)
    print(f"Dataset preflight passed: {config.name} ({sample_count} test images).")

# Direct Google Drive download is enabled by default. In Kaggle, turn on
# Settings -> Internet before running this notebook. Set this to False only
# when using an attached Kaggle dataset containing the selected checkpoints.
DOWNLOAD_FROM_GOOGLE_DRIVE = True
ATTACHED_CHECKPOINT_ROOT = None
DOWNLOAD_ROOT = Path("/kaggle/working/inpformer_checkpoints")

# =============================================================================
# USER-CONTROLLABLE CORRUPTION AND EXECUTION SETTINGS
# =============================================================================
USE_CATEGORIZED_CORRUPTIONS = True
CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise",
    "shot_noise",
    "impulse_noise",
    "defocus_blur",
    "motion_blur",
    "zoom_blur",
    "brightness",
    "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = [
    "noise",
    "blur",
    "photometric",
    "geometric",
]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS
    else UNCATEGORIZED_CORRUPTION_TYPES
)
# Set the selected list above to [] for a clean-only run. Severity 0 is
# reserved for the clean baseline and must not appear here.
INCLUDE_CLEAN_BASELINE = True
SEVERITY_LEVELS = [1, 2, 3, 4]
# Default shots are 1/2/4. Use one value per Kaggle session when runtime is
# limited. Evaluation count is len(SHOTS_TO_RUN) x len(DATASETS_TO_RUN).
SHOTS_TO_RUN = [1, 2, 4]
DEVICE = "cuda"  # Change to "cpu" only for a deliberate CPU run.
BATCH_SIZE = 4
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        "DEVICE requests CUDA, but no GPU is available. In Kaggle, open "
        "Settings -> Accelerator and select a GPU, or explicitly use CPU."
    )
# Keep this as None for large runs. The complete both-dataset categorized
# suite has 62,192 corrupted image/condition pairs per shot.
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"
STRICT_SOURCE_COMMIT = True

def google_drive_id(url):
    return url.split("/d/", 1)[1].split("/", 1)[0]

MIN_CHECKPOINT_BYTES = 100 * 1024 * 1024

def valid_torch_checkpoint_file(path):
    if not path.is_file() or path.stat().st_size < MIN_CHECKPOINT_BYTES:
        return False
    if not zipfile.is_zipfile(path):
        return False
    with zipfile.ZipFile(path) as checkpoint_zip:
        if checkpoint_zip.testzip() is not None:
            return False
        members = checkpoint_zip.namelist()
    return any(name.endswith("/data.pkl") for name in members)

def download_official_suite(root, shots, datasets):
    for shot in shots:
        for dataset in datasets:
            destination = root / official_checkpoint_directory(shot, dataset) / "model.pth"
            if valid_torch_checkpoint_file(destination):
                print(f"Reusing {destination}")
                continue
            if destination.exists():
                print(f"Removing incomplete checkpoint: {destination}")
                destination.unlink()
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_suffix(destination.suffix + ".download")
            url = OFFICIAL_CHECKPOINT_URLS[shot][dataset]
            print(f"Downloading official {shot}-shot {dataset} checkpoint...")
            try:
                result = gdown.download(
                    id=google_drive_id(url),
                    output=str(temporary),
                    quiet=False,
                    resume=True,
                )
            except Exception as exc:
                raise RuntimeError(
                    "Google Drive download failed. Enable Internet in the Kaggle "
                    "notebook settings. If Google Drive reports a quota limit, "
                    "attach the selected checkpoints as a Kaggle dataset and set "
                    "DOWNLOAD_FROM_GOOGLE_DRIVE=False. "
                    f"URL: {url}"
                ) from exc
            if result is None or not valid_torch_checkpoint_file(temporary):
                raise RuntimeError(
                    "Google Drive returned an incomplete/non-checkpoint file. "
                    "Check Kaggle Internet and Google Drive quota, then rerun to "
                    f"resume. URL: {url}; temporary file: {temporary}"
                )
            temporary.replace(destination)
    return root

if DOWNLOAD_FROM_GOOGLE_DRIVE:
    checkpoint_root = download_official_suite(
        DOWNLOAD_ROOT, SHOTS_TO_RUN, DATASETS_TO_RUN
    )
else:
    if not ATTACHED_CHECKPOINT_ROOT:
        raise ValueError("Set ATTACHED_CHECKPOINT_ROOT when direct download is disabled.")
    checkpoint_root = Path(ATTACHED_CHECKPOINT_ROOT)
CHECKPOINT_PATHS = discover_official_checkpoints(
    str(checkpoint_root), shots=SHOTS_TO_RUN, datasets=DATASETS_TO_RUN
)

print(
    f"\n====== LAUNCHING {len(SHOTS_TO_RUN)} SHOT SETTING(S) x "
    f"{len(DATASETS_TO_RUN)} DATASET(S) = "
    f"{len(SHOTS_TO_RUN) * len(DATASETS_TO_RUN)} EVALUATIONS ======"
)
print(f"Evaluation datasets: {DATASETS_TO_RUN}")
print("Checkpoint pairing: same-dataset official few-shot weights")
for dataset_name in DATASETS_TO_RUN:
    print(f"{DATASET_LABELS[dataset_name]}: {DATASET_ROOTS[dataset_name]}")
print(f"Checkpoints: {checkpoint_root}")
print(f"Clean baseline: {INCLUDE_CLEAN_BASELINE}")
print(f"Corruption mode: {'categorized' if USE_CATEGORIZED_CORRUPTIONS else 'uncategorized'}")
print(f"Corruptions: {CORRUPTION_TYPES}")
print(f"Severities: {SEVERITY_LEVELS}")
print(f"Corruption seed: {CORRUPTION_SEED}")
print(f"Device/batch: {DEVICE} / {BATCH_SIZE}")
print(f"Outputs: {OUTPUT_ROOT}")

run_official_evaluations(
    mvtec_root=MVTEC_ROOT,
    visa_root=VISA_ROOT,
    output_root=OUTPUT_ROOT,
    inpformer_root=str(INPFORMER_ROOT),
    checkpoint_paths=CHECKPOINT_PATHS,
    shots=SHOTS_TO_RUN,
    datasets=DATASETS_TO_RUN,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    corruption_seed=CORRUPTION_SEED,
    include_clean=INCLUDE_CLEAN_BASELINE,
    strict_source_commit=STRICT_SOURCE_COMMIT,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
archives = [f"INP-Former-{shot}-shot_artifacts.zip" for shot in SHOTS_TO_RUN]
print(f"Finished. Collect {len(archives)} archive(s): {archives}")
